# Signpost-D, part 1: training

Signpost-D answers typed decision questions about one case (a text or JSON "state"): true/false statements, choices between labelled options, and scores on ordered scales. For each question it returns a probability for every option, with no generated text. This notebook trains it; `02_evaluate.ipynb` serves it with vLLM and scores it exactly as the benchmark does.

- **Format: vLLM's own.** The benchmark's scorer serves a model with vLLM and asks `POST /v1/systemone`. For a Qwen model, vLLM sends **one request per question**: the state, then the question with labelled options (`A`, `B`, … for a choice; `yes` / `no` for a true/false statement; `0`, `1`, … for a score), then one line saying how to answer. The answer is the probability of each label as the **first token of the reply**, softmaxed over the labels. This notebook trains on exactly that prompt and that reading, so what is trained is what is scored.
- **Model:** Qwen3.5-4B (its text model), fine-tuned with LoRA rank 16 (α = 16) on the attention, MLP and linear-attention projections. Base weights in fp16, LoRA weights in fp32, fp16 autocast.
- **Loss:** soft-label cross-entropy against the judges' averaged distribution, plus, for score questions, the distance between the expected levels.
- **Data:** the Hugging Face dataset `DATASET`: 2,575 generated records (12,274 questions) to train on, and 200 records from other domains as dev.
- **Recipe:** LR 1e-4, warm-up 15%, linear decay, gradient clipping 1.0, two cases (all their questions) per optimizer step, 2 epochs, three seeds. `02_evaluate.ipynb` merges the three seeds into the final model.
- **Hardware:** one GPU (a Colab G4 trains a seed in about 37 minutes), or two Kaggle T4s (data-parallel through `accelerate`'s `notebook_launcher`; with one GPU, two cases are accumulated per step, so the steps and schedule are the same).
- **Restart:** a checkpoint every 200 steps and after every epoch in `checkpoints/`; with `CKPT_REPO` set, on Colab the folder is also pushed to that private Hugging Face repo. "Run all" again resumes, and finished seeds are read back.


## Parameters

Everything a run depends on. Finished seeds are read back, so running the notebook again only trains what is missing.


In [ ]:
# LoRA rank (alpha = rank), learning rate, questions per scoring pass
MODEL = {"name": "Qwen/Qwen3.5-4B", "r": 16, "lr": 1e-4, "eval_batch": 16}
SEEDS = [42, 1042, 2042]              # one run per seed, one after another; finished seeds are read back
DATASET = "skamalj/signpost-d-decisions"   # the training data: train (2,575 records) and dev (200 records)
EPOCHS = 2
EPOCH_KL_TOL = 0.005                  # the last epoch is kept unless its dev KL is worse than the best by more than this
WARMUP_FRAC = 0.15
LORA_DROPOUT = 0.05
SCORE_DIST_W = 0.5                    # weight of the expected-level term for score questions
COLLAPSE_STEP, COLLAPSE_MARGIN, ATTEMPTS = 400, 0.02, 2   # a run whose dev KL stays near uniform is restarted with another seed
CKPT_EVERY = 200                      # optimizer steps between mid-epoch checkpoints
CASES_PER_STEP = 2                    # cases per optimizer step on any GPU count: 2 GPUs x 1, or 1 GPU x 2 accumulated
GPUS = 2                              # data-parallel over this many GPUs, fewer if fewer are visible
CKPT_REPO = None                      # a private model repo: on Colab, checkpoints/ is fetched from it and pushed back
FAST_CONV = False                     # install causal-conv1d, the fast kernel for the linear-attention layers' short
                                      # convolution (built from source if no wheel fits; skipped if that fails)
RUN_NAME = "01_train"


## Packages

Pinned `transformers` and `peft`, and the Triton kernels for Qwen3.5's linear-attention layers (`fla-core`). Without them Qwen3.5 still runs, on a much slower reference path. With `FAST_CONV`, also `causal-conv1d`, the fast kernel for the short convolution inside those layers; without it transformers uses a correct but slower PyTorch version. If no ready-made wheel fits, pip builds it from source (a few minutes); if that fails, the run carries on without it.

In [ ]:
!pip uninstall -y -q torchao      # an old torchao on Kaggle makes peft refuse to load; it is not used
!pip install -q --force-reinstall --no-deps "transformers==5.17.0" "peft==0.21.2"
!pip install -q --no-deps flash-linear-attention fla-core   # Qwen3.5's linear-attention kernels (Triton)

import datetime, gc, getpass, importlib.util, json, math, os, random, statistics, subprocess, sys, time

if FAST_CONV and importlib.util.find_spec("causal_conv1d") is None:
    try:
        built = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "causal-conv1d"],
                               capture_output=True, text=True, timeout=1800)
        print("causal-conv1d installed" if built.returncode == 0 else f"causal-conv1d NOT installed: {built.stderr[-300:]}")
    except subprocess.TimeoutExpired:
        print("causal-conv1d NOT installed: the build took over 30 minutes")
    importlib.invalidate_caches()        # so the new package is found without a restart

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ["PYTORCH_NVML_BASED_CUDA_CHECK"] = "1"   # CUDA must not start before the GPU processes are forked
from collections import Counter
from importlib.metadata import version
from pathlib import Path

import torch
import torch.distributed as dist
import torch.nn.functional as F
import transformers
from huggingface_hub import HfApi

## The Hugging Face token

Needed to read the dataset and, with `CKPT_REPO`, to fetch and push the checkpoints. It is read from the environment, a Kaggle secret or a Colab secret, and on Colab, if none is set, asked for in a hidden prompt. Only where it came from is printed, never the token. A second check reports whether the fast linear-attention kernels load.


In [ ]:
ON_COLAB = importlib.util.find_spec("google.colab") is not None


def hf_token():
    """-> (token or None, where it came from): the environment, a Kaggle secret, a Colab secret, or (on Colab) a prompt."""
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"], "the environment"
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN"), "Kaggle secret 'HF_TOKEN'"
    except Exception:
        pass
    if ON_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
            if token:
                return token, "Colab secret 'HF_TOKEN'"
        except Exception:
            pass
        token = getpass.getpass("Hugging Face token: ").strip()
        return token or None, "the prompt"
    return None, "NOT FOUND"


HF_TOKEN, token_from = hf_token()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN                # every Hugging Face call (and every GPU process) uses it
print(f"Hugging Face token: {token_from}")          # where it came from, never the token itself
fla_fast = subprocess.run([sys.executable, "-c", "from fla.ops.gated_delta_rule import chunk_gated_delta_rule"],
                          capture_output=True).returncode == 0   # checked in a child process: no CUDA in this one
print(f"linear-attention kernels: {'fast (fla-core)' if fla_fast else 'NOT available, the slow reference path will be used'}")
conv_fast = subprocess.run([sys.executable, "-c", "from causal_conv1d import causal_conv1d_fn"], capture_output=True).returncode == 0
print(f"short-convolution kernel: {'fast (causal-conv1d)' if conv_fast else 'reference PyTorch path (correct, slower)'}")

## Checkpoints, device and progress log

`checkpoints/` holds every run: resume points, adapters, results. With `CKPT_REPO` set, on a fresh Colab machine it is first downloaded from there, and after every epoch and finished seed the changed files are pushed back, so a lost machine resumes where it stopped. The log goes to the screen and to `01_train_run.log`.


In [ ]:
MAX_PROMPT_TOKENS = 8191                            # one question; vLLM serves 8,192 tokens including the answer
CKPT = Path("checkpoints")
CKPT.mkdir(exist_ok=True)
if ON_COLAB and CKPT_REPO and not any(CKPT.iterdir()):   # a fresh Colab machine: fetch the runs so far
    from huggingface_hub import snapshot_download
    snapshot_download(CKPT_REPO, repo_type="model", local_dir=str(CKPT), token=HF_TOKEN)
    print(f"checkpoints fetched from {CKPT_REPO}")

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")   # each process sets its own GPU in the last cell
RANK, WORLD = 0, 1                                  # this process and the number of processes (set again in each process)
AMP = DEVICE.type == "cuda"
RUN_LOG = Path(f"{RUN_NAME}_run.log")
assert (transformers.__version__, version("peft")) == ("5.17.0", "0.21.2"), "restart the kernel and run from the top"
api = HfApi()


def log(message):
    if RANK:                                        # one process writes the log
        return
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(RUN_LOG, "a") as fh:
        fh.write(line + "\n")


def gpu():
    if not AMP:
        return "CPU"
    used, peak = torch.cuda.memory_allocated(DEVICE) / 1e9, torch.cuda.max_memory_allocated(DEVICE) / 1e9
    return f"GPU {used:.1f} GB (peak {peak:.1f})"


def sync_checkpoints(note):
    """On Colab, push checkpoints/ to CKPT_REPO (only changed files travel), so a lost machine resumes from there.
    Never stops a run."""
    if not (ON_COLAB and CKPT_REPO) or RANK:
        return
    try:
        api.upload_folder(folder_path=str(CKPT), repo_id=CKPT_REPO, repo_type="model", ignore_patterns=["*.tmp"],
                          commit_message=note, token=HF_TOKEN)
        log(f"  checkpoints pushed to {CKPT_REPO} ({note})")
    except Exception as e:
        log(f"  WARNING: checkpoint push failed ({type(e).__name__}: {str(e)[:120]}); the local copy is intact")


log(f"{RUN_NAME} | {MODEL['name']} | seeds {SEEDS} | data {DATASET} | torch {torch.__version__} | "
    f"GPUs visible {torch.cuda.device_count()} (using up to {GPUS})")

## The data

The records come from the Hugging Face dataset `DATASET`. Each line is one record in System One form: a state, its questions, and the gold: the judges' averaged probability for every option.


In [ ]:
from huggingface_hub import hf_hub_download


def load_jsonl(path):
    return [json.loads(line) for line in open(path, encoding="utf-8")]


def dataset_split(split):
    """One split of DATASET (train or dev) as a list of records."""
    return load_jsonl(hf_hub_download(DATASET, f"data/{split}.jsonl", repo_type="dataset", token=HF_TOKEN))


## One item per question

`render` turns a question's criteria into options exactly as vLLM does:
- **true/false:** always two options, `yes` (with the "true" description) then `no` (with the "false" description);
- **choice:** the criteria's option names as written (underscores kept), each with its description;
- **score:** the level strings, lowest first.

`items_of` makes one item per question: the state (JSON written compactly, as vLLM writes it), the question, its options and the target distribution, keyed by the option names (the gold's `true` / `false` become `yes` / `no`).

In [ ]:
def render(q):
    """A System One question -> ({option name: description or None} in vLLM's order, the gold's keys in the same order):
    true/false as yes / no, choice by its names as written, score by its level strings, lowest first ("0".."n-1")."""
    criteria = q.get("criteria")
    if q["type"] == "noul":
        criteria = criteria or {}
        options = {"yes": criteria.get("true"), "no": criteria.get("false")}
        keys = ["true", "false"]
    elif q["type"] == "choice":
        options = {str(k): None if v is None else str(v) for k, v in criteria.items()}
        keys = list(criteria)
    else:
        options = {str(level): None for level in criteria}
        keys = [str(i) for i in range(len(criteria))]
    return options, keys


def items_of(case):
    """A case -> one item per question: the state, the question, its options and the target distribution."""
    state = case["state"] if isinstance(case["state"], str) else json.dumps(case["state"], ensure_ascii=False)
    out = []
    for qid, q in case["questions"].items():
        options, keys = render(q)
        gold = case["gold"][qid]["probabilities"]
        out.append({"task": "d_" + q["type"], "id": case["id"], "qid": qid, "state": state,
                    "question": "" if q.get("instructions") is None else str(q["instructions"]),
                    "options": options, "levels": list(options) if q["type"] == "score" else None,
                    "soft": {name: gold[key] for name, key in zip(options, keys)}})
    return out

## Training and dev sets

All of `train` is trained on. `dev` (200 records from 8 domains that are not in `train`) is never trained on: it checks for collapse and chooses the epoch.


In [ ]:
train_cases, dev_cases = dataset_split("train"), dataset_split("dev")
train_d = [it for c in train_cases for it in items_of(c)]
dev_d = [it for c in dev_cases for it in items_of(c)]
KIND = {(c["id"], q): f"{c['domain']}/{q}" for c in train_cases + dev_cases for q in c["questions"]}
print(f"train: {len(train_cases)} records, {len(train_d)} questions {dict(Counter(it['task'] for it in train_d))} | "
      f"dev: {len(dev_cases)} records, {len(dev_d)} questions")


## The prompt: vLLM's text for one question

This is vLLM's own wording (`vllm/entrypoints/openai/decisions/question_types.py` and `strategies.py`, the same in release 0.32.0 and in the nightly the benchmark installs). The user turn is the state, a blank line, then:

```
Question: <the question>
A: <option name> - <description>        (choice; a missing description leaves "A: <option name>")
yes: <what true means>                  (true/false; "yes" alone without a description, then "no" the same way)
0: <level>                              (score; levels lowest first, numbered from 0)
Answer with the letter of one option only.      (choice)
Answer with yes or no only.                     (true/false)
Answer with the number of one level only.       (score)
```

There is no system message (the benchmark sends no `instructions`). Each question is its own request, so no question sees another.

In [ ]:
LABELS = {"d_choice": [chr(65 + i) for i in range(26)],   # vLLM's labels, in option order
          "d_noul": ["yes", "no"],
          "d_score": [str(i) for i in range(10)]}
REPLY = {"d_choice": "Answer with the letter of one option only.",
         "d_noul": "Answer with yes or no only.",
         "d_score": "Answer with the number of one level only."}


def question_text(item):
    """The question as vLLM writes it after the state: the question, one line per labelled option, the reply line."""
    lines = [f"Question: {item['question']}"] if item["question"] else []
    for label, (name, description) in zip(LABELS[item["task"]], item["options"].items()):
        if item["task"] == "d_noul":                  # the side's description, or the bare label
            lines.append(f"{label}: {description}" if description else label)
        elif item["task"] == "d_choice":
            lines.append(f"{label}: {name} - {description}" if description else f"{label}: {name}")
        else:
            lines.append(f"{label}: {name}")
    lines.append(REPLY[item["task"]])
    return "\n".join(lines)


def user_turn(item):
    """vLLM's user message: the state, a blank line, the question."""
    return f"{item['state']}\n\n{question_text(item)}"


def by_case(items):
    """Consecutive items of one case -> [[item, ...], ...] in order."""
    groups = []
    for it in items:
        if groups and groups[-1][0]["id"] == it["id"]:
            groups[-1].append(it)
        else:
            groups.append([it])
    return groups

## Loading the base model

Qwen3.5-4B from Hugging Face, pinned to the revision current at load time, in fp16 (half the memory of fp32, and what a T4 runs fast). Only the text model is loaded, not the vision tower. For training, gradient checkpointing recomputes activations in the backward pass instead of storing them: slower, but a 4B model then fits with long cases.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer


def load_base(name, train):
    """-> (tokenizer, model in fp16, revision). For training, gradient checkpointing: activations are recomputed in the
    backward pass instead of stored, which trades some speed for memory."""
    rev = api.model_info(name, token=HF_TOKEN).sha
    tok = AutoTokenizer.from_pretrained(name, revision=rev, token=HF_TOKEN)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(name, revision=rev, token=HF_TOKEN, dtype=torch.float16,
                                             device_map={"": DEVICE.index or 0} if AMP else None)
    m.config.use_cache = False
    if train:
        m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    return tok, m, rev

## LoRA: what is trained

The 4.2 B base weights stay frozen. LoRA adds to each chosen projection two small matrices, A (rank × input) and B (output × rank), whose product B·A is a low-rank change to that weight; only A and B are trained: about 30 M parameters, 0.7% of the model. Rank 16 with alpha 16 means the change is added at scale 1. The targets are the attention projections, the MLP and Qwen3.5's linear-attention projections; the vision tower is excluded.

In [ ]:
LORA_TARGETS = r"^(?!.*(visual|vision)).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj|in_proj_qkv|in_proj_z|out_proj)$"


def add_lora(m, rank):
    """Freezes the base and adds LoRA (alpha = rank) to every projection named in LORA_TARGETS; the LoRA weights are
    kept in fp32, so small updates are not lost to fp16 rounding."""
    from peft import LoraConfig, get_peft_model       # imported here: importing peft early would start CUDA
    m = get_peft_model(m, LoraConfig(r=rank, lora_alpha=rank, lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGETS))
    for p in m.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    return m

## The answer is the reply's first token

vLLM ends every prompt with the chat template's generation prompt (thinking off), and reads the next token: the first token of the reply. It works out each label's token the way `label_token_ids` does here: it renders a probe chat, keeps the text after its last special token (the end of the generation prompt), and tokenizes that text with the label appended. The label must add exactly **one** new token there, different from every other label's; vLLM refuses to serve a model where that fails, so this notebook stops too.

In [ ]:
def generation_tail(tok):
    """The ids after the last special token of a probe chat prompt (thinking off): how every prompt ends (vLLM's
    prompt_tail)."""
    probe = tok.apply_chat_template([{"role": "user", "content": "x"}], add_generation_prompt=True, tokenize=False,
                                    enable_thinking=False)
    ids = tok(probe, add_special_tokens=False)["input_ids"]
    added = set(tok.get_added_vocab().values())
    start = max((i + 1 for i, t in enumerate(ids) if t in added), default=0)
    return ids[start:]


def label_token_ids(tok):
    """{type: [token id of each label as the reply's first token]}, as vLLM's reply_label_ids works them out. Each label
    must add exactly one token after the prompt's tail, distinct from the other labels of its type."""
    tail = generation_tail(tok)
    text = tok.decode(tail)
    out = {}
    for task, labels in LABELS.items():
        ids = []
        for label in labels:
            extended = tok(text + label, add_special_tokens=False)["input_ids"]
            assert extended[:-1] == tail and extended[-1] not in ids, f"label {label!r} is not one distinct token"
            ids.append(extended[-1])
        out[task] = ids
    return out

## One question as one sequence

The chat template wraps the user turn and adds the generation prompt with thinking off, as vLLM does (`add_generation_prompt=True`, `enable_thinking=False`, no extra special tokens). The answer is read at the sequence's last position: the model's prediction there is the reply's first token. A case's questions are separate sequences that repeat the state; they are run side by side in one batch.

In [ ]:
def question_ids(tok, item):
    """One question as vLLM sends it -> its token ids; the answer is the next token after the last one."""
    prompt = tok.apply_chat_template([{"role": "user", "content": user_turn(item)}], add_generation_prompt=True,
                                     tokenize=False, enable_thinking=False)
    return tok(prompt, add_special_tokens=False)["input_ids"]

## Reading the answers

A batch of questions is padded to one length (padding on the right, masked) and run through the transformer once. At each question's last real position, the final hidden state is multiplied by the output embeddings of that question's label tokens only: one logit per option. A softmax over them is exactly vLLM's answer, because vLLM softmaxes the labels' log-probabilities, and those differ from the logits by the same constant for every label. Questions with fewer options get −10,000 in the unused places, so softmax gives them zero.

In [ ]:
def label_logits(model, items):
    """One sequence per question, padded to one length and run in one forward pass; at each sequence's last position the
    hidden state is multiplied by the output embeddings of its labels only -> [items, most options] logits, missing
    options -1e4."""
    built = [question_ids(model.tok, it) for it in items]
    length = max(len(x) for x in built)
    assert length <= MAX_PROMPT_TOKENS, f"a question of {length} tokens"
    pad = model.tok.pad_token_id
    ids = torch.tensor([x + [pad] * (length - len(x)) for x in built], device=DEVICE)
    mask = torch.tensor([[1] * len(x) + [0] * (length - len(x)) for x in built], device=DEVICE)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
        hidden = model.inner(input_ids=ids, attention_mask=mask).last_hidden_state
    last = hidden[torch.arange(len(built), device=DEVICE), torch.tensor([len(x) - 1 for x in built], device=DEVICE)]
    K = max(len(it["options"]) for it in items)
    label = torch.tensor([model.label_ids[it["task"]][:len(it["options"])] + [0] * (K - len(it["options"])) for it in items],
                         device=DEVICE)
    logits = torch.einsum("nh,nkh->nk", last.float(), model.head_w[label].float())
    used = torch.tensor([[j < len(it["options"]) for j in range(K)] for it in items], device=DEVICE)
    return logits.masked_fill(~used, -1e4)

## The model object

Ties the pieces together: the tokenizer, the model (with LoRA when training), its inner transformer, the output embeddings and each type's label tokens. On loading it logs how the prompt ends and which tokens the labels are, and shows one dev question exactly as the model reads it. `scores` reads a batch; `lora_params` lists the trainable weights.

In [ ]:
class DecoderModel:
    """The pieces above together: the base (with LoRA when training), its inner transformer (the part before the
    output layer), the output embeddings, each type's label tokens, and scores() for a batch of items."""

    shown = False                                      # one example prompt per notebook run

    def __init__(self, cfg, train):
        self.cfg = cfg
        self.tok, m, self.rev = load_base(cfg["name"], train)
        self.m = add_lora(m, cfg["r"]) if train else m
        base = self.m.get_base_model() if train else self.m
        self.inner = base.model.language_model if hasattr(base.model, "language_model") else base.model
        self.head_w = self.m.get_output_embeddings().weight
        self.label_ids = label_token_ids(self.tok)
        n_train = sum(p.numel() for p in self.m.parameters() if p.requires_grad) if train else 0
        log(f"loaded {cfg['name']} @ {self.rev[:8]} | fp16 | trainable {n_train / 1e6:.1f} M | {gpu()}")
        log(f"prompt ends {self.tok.decode(generation_tail(self.tok))!r} | label tokens: "
            + " | ".join(f"{t[2:]} {LABELS[t][:3]} -> {ids[:3]}" for t, ids in self.label_ids.items()))
        if not DecoderModel.shown and RANK == 0:
            example = next(it for it in dev_d if it["task"] == "d_score")
            print("----- one dev question as the model reads it (state cut) -----")
            text = self.tok.decode(question_ids(self.tok, example))
            print(text[:300] + "\n[...]\n" + text[-600:])
            DecoderModel.shown = True

    def scores(self, items):
        return label_logits(self, items)

    def lora_params(self):
        return [p for p in self.m.parameters() if p.requires_grad]

    def train(self, mode=True):
        self.m.train(mode)

    def free(self):
        self.m = self.inner = self.head_w = None
        gc.collect()
        if AMP:
            torch.cuda.empty_cache()

## The loss

Cross-entropy against the judges' distribution, so the model learns how sure to be as well as which option to pick. Score questions add a small term for the distance between the predicted and the target expected level, so a near miss costs less than a far one.

In [ ]:
def soft_loss(scores, items):
    """Cross-entropy against the judges' distribution. Score questions add SCORE_DIST_W x |expected level - target
    expected level| / (levels - 1), each option weighted by its level in the rubric."""
    K = scores.shape[1]
    target = torch.tensor([[it["soft"][n] for n in it["options"]] + [0.0] * (K - len(it["options"])) for it in items],
                          device=scores.device)
    target = target / target.sum(1, keepdim=True)
    logp = F.log_softmax(scores, dim=1)
    loss = -(target * logp).sum(1).mean()
    is_score = torch.tensor([it["task"] == "d_score" for it in items], device=scores.device)
    if is_score.any():
        level = torch.tensor([[it["levels"].index(n) if it["levels"] else 0 for n in it["options"]]
                              + [0] * (K - len(it["options"])) for it in items], device=scores.device, dtype=scores.dtype)
        n_levels = torch.tensor([len(it["options"]) for it in items], device=scores.device, dtype=scores.dtype)
        gap = ((logp.exp() * level).sum(1) - (target * level).sum(1)).abs() / (n_levels - 1).clamp(min=1)
        loss = loss + SCORE_DIST_W * gap[is_score].mean()
    return loss

## Scoring over the GPUs

Questions are scored `eval_batch` at a time. With two GPUs each takes every other batch and the results are gathered back in order. `predictions` returns, for every question, the probability of each option.

In [ ]:
def barrier():
    if WORLD > 1:
        dist.barrier()


def spread(jobs, fn):
    """fn(list of jobs) -> one result per job. Every WORLD-th job goes to each GPU; the results come back in order on all."""
    mine = list(range(RANK, len(jobs), WORLD))
    results = fn([jobs[i] for i in mine])
    if WORLD == 1:
        return results
    parts = [None] * WORLD
    dist.all_gather_object(parts, list(zip(mine, results)))
    out = [None] * len(jobs)
    for part in parts:
        for i, r in part:
            out[i] = r
    return out


def predictions(model, items, batch=16):
    """-> [(item, {option: probability})] in order; batch = questions per forward pass."""
    model.train(False)
    chunks = [items[s:s + batch] for s in range(0, len(items), batch)]

    def run(cs):
        out = []
        with torch.no_grad():
            for chunk in cs:
                rows = torch.softmax(model.scores(chunk), dim=1).float().cpu().tolist()
                out.append([{n: row[j] for j, n in enumerate(it["options"])} for it, row in zip(chunk, rows)])
        return out

    return [(it, p) for chunk, ps in zip(chunks, spread(chunks, run)) for it, p in zip(chunk, ps)]

## The measures

The benchmark's four measures, against our own labels: accuracy (the top option equals the gold's top option), KL from the gold, Brier and 10-bin ECE. Before measuring, one temperature per question type is fitted on our dev set, which changes confidence only, never the top option.

In [ ]:
def soften(p, temperature):
    """{option: probability} softened as p ** (1 / T), renormalised."""
    q = {n: max(v, 1e-12) ** (1 / temperature) for n, v in p.items()}
    z = sum(q.values())
    return {n: v / z for n, v in q.items()}


def metrics(preds, temps=None):
    """Accuracy (top option = the label's top option), KL from the label, Brier, ECE (10 bins); overall and per type.
    temps: one temperature per question type ({"d_noul": T, ...}), or None."""
    rows = []
    for it, p in preds:
        q = soften(p, (temps or {}).get(it["task"], 1.0))
        g = it["soft"]
        rows.append({"task": it["task"], "ok": max(q, key=q.get) == max(g, key=g.get), "conf": max(q.values()),
                     "kl": sum(gv * math.log(gv / max(q[n], 1e-9)) for n, gv in g.items() if gv > 0),
                     "brier": sum((q[n] - g[n]) ** 2 for n in g)})

    def summary(rs):
        bins = [[0, 0, 0.0] for _ in range(10)]
        for r in rs:
            b = bins[min(int(r["conf"] * 10), 9)]
            b[0] += 1; b[1] += r["ok"]; b[2] += r["conf"]
        ece = sum(b[0] / len(rs) * abs(b[1] / b[0] - b[2] / b[0]) for b in bins if b[0])
        return {"n": len(rs), "accuracy": round(statistics.mean(r["ok"] for r in rs), 4),
                "kl": round(statistics.mean(r["kl"] for r in rs), 4),
                "brier": round(statistics.mean(r["brier"] for r in rs), 4), "ece": round(ece, 4)}

    return {"all": summary(rows), **{t: summary([r for r in rows if r["task"] == t]) for t in sorted({r["task"] for r in rows})}}


def fit_temperatures(preds):
    """One temperature per question type, each minimising KL on our own dev questions of that type."""
    grid = [0.5, 0.75, 1, 1.25, 1.5, 2, 2.5, 3, 4, 5]
    temps = {}
    for task in sorted({it["task"] for it, _ in preds}):
        sub = [(it, p) for it, p in preds if it["task"] == task]
        temps[task] = min(grid, key=lambda t: metrics(sub, {task: t})["all"]["kl"])
    return temps


def by_group(preds, temps, key):
    """Accuracy and KL per group (key(item) -> group name)."""
    names = sorted({key(it) for it, _ in preds})
    return {g: metrics([(it, p) for it, p in preds if key(it) == g], temps)["all"] for g in names}

## A collapse guard

A run whose predictions stay near a uniform guess, or whose loss stops being a number (an fp16 overflow), is stopped early and restarted with another seed instead of wasting hours. `uniform_kl` is the dev KL a uniform guess would get: the bar a working model must clearly beat.

In [ ]:
class Collapsed(Exception):
    """The predictions stayed at a uniform guess, or the loss stopped being a number."""


def uniform_kl(items):
    return statistics.mean(sum(g * math.log(g * len(it["soft"])) for g in it["soft"].values() if g > 0) for it in items)

## Training order

Every epoch sees every training case once, in an order shuffled with the seed and epoch; a case's questions are trained together. The options of a choice question are shuffled too, so no letter carries a fixed answer. True/false keeps `yes` then `no` (vLLM always writes them in that order) and score levels keep their order (their order is their meaning). All of it is seeded per case and epoch, so a run resumed mid-epoch sees exactly the same order.

In [ ]:
def training_order(seed, epoch, acc):
    """All training cases (one case per micro-batch) in one order, the same on every GPU: shuffled with seed + epoch, and
    cut to a whole number of optimizer steps (WORLD GPUs x acc micro-batches each). Micro-batch k goes to GPU k % WORLD."""
    cases = by_case(train_d)
    random.Random(seed + epoch).shuffle(cases)
    return cases[: len(cases) // (WORLD * acc) * WORLD * acc]


def prepare(case, seed, epoch):
    """A case as the model sees it this epoch: the options of its choice questions shuffled (true/false and score keep
    vLLM's order). Seeded per case and epoch, so a resumed run repeats it exactly."""
    rng = random.Random(f"{seed}-{epoch}-{case[0]['id']}")
    case = [dict(it) for it in case]
    for it in case:
        if it["task"] == "d_choice":
            names = list(it["options"])
            rng.shuffle(names)
            it["options"] = {n: it["options"][n] for n in names}
    return case

## Optimizer and learning-rate schedule

**AdamW** keeps a running mean and spread of each weight's gradients and scales each update by them, with weight decay 0.01. The **learning rate** (1e-4) is reached by a linear warm-up over the first 15% of the steps and then falls linearly to zero at the last step: big steps early, small settling steps at the end. The **gradient scaler** is needed for fp16: it multiplies the loss before the backward pass so tiny gradients do not round to zero, divides it out before the update, and skips a step whose gradients overflowed.

In [ ]:
def make_optimizer(params, lr, total):
    """AdamW on the LoRA weights (weight decay 0.01). The learning rate rises linearly over the first WARMUP_FRAC of the
    steps, then falls linearly to zero at the last step. The gradient scaler multiplies the loss before the backward pass
    so small fp16 gradients do not round to zero, and divides it out again before the update."""
    warm = max(1, int(WARMUP_FRAC * total))
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    schedule = lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, schedule)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    return opt, sched, scaler

## The training state

Everything a run carries between steps: the step count, the dev history, the losses and gradient norms, the epoch kept so far, and a counter for broken losses. It is also what a resume point restores.

In [ ]:
class TrainState:
    """What a run carries from step to step, and what a resume point must restore."""

    def __init__(self):
        self.step = 0                                    # optimizer steps done
        self.history = []                                # dev measures after each epoch
        self.losses = []                                 # loss of every micro-batch (the last 500 are saved)
        self.norms = []                                  # gradient norm before clipping, every step
        self.best = (float("inf"), None, 0)              # (dev KL, LoRA weights, epoch) of the epoch kept so far
        self.min_kl = float("inf")                       # the lowest dev KL seen
        self.start_epoch = 1
        self.bad = 0                                     # micro-batches in a row whose loss was not a number

## Saving and resuming

A resume point holds the LoRA weights, the optimizer's moments, the schedule's position, the scaler's state and the training state, enough to continue as if never stopped. One is written every 200 steps and after every epoch, through a temporary file, so an interrupted save never leaves a broken checkpoint.

In [ ]:
def save_checkpoint(ckdir, model, opt, sched, scaler, st, epoch, complete):
    """A resume point: the LoRA weights, the optimizer, schedule and scaler states, and the run's state. Written to a
    temporary file first, so an interruption never leaves a half-written checkpoint."""
    if RANK == 0:
        ckdir.mkdir(parents=True, exist_ok=True)
        lora = {n: p.detach().cpu() for n, p in model.m.named_parameters() if p.requires_grad}
        torch.save({"epoch": epoch, "complete": complete, "step": st.step, "history": st.history, "run": st.losses[-500:],
                    "lora": lora, "opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(),
                    "best_kl": st.best[0], "best_lora": st.best[1], "best_epoch": st.best[2], "min_kl": st.min_kl},
                   ckdir / "latest.pt.tmp")
        os.replace(ckdir / "latest.pt.tmp", ckdir / "latest.pt")
        log(f"  checkpoint saved at step {st.step} (epoch {epoch}{' complete' if complete else ''})")
    barrier()


def load_checkpoint(ckdir, model, opt, sched, scaler, st):
    """Restores the last resume point, if there is one; the run continues mid-epoch or after a finished epoch."""
    latest = ckdir / "latest.pt"
    if not latest.exists():
        return
    saved = torch.load(latest, map_location="cpu", weights_only=False)
    by_name = dict(model.m.named_parameters())
    for n, v in saved["lora"].items():
        by_name[n].data.copy_(v)
    opt.load_state_dict(saved["opt"]); sched.load_state_dict(saved["sched"]); scaler.load_state_dict(saved["scaler"])
    st.step, st.history, st.losses = saved["step"], saved["history"], saved["run"]
    st.best, st.min_kl = (saved["best_kl"], saved["best_lora"], saved["best_epoch"]), saved.get("min_kl", saved["best_kl"])
    st.start_epoch = saved["epoch"] + 1 if saved["complete"] else saved["epoch"]
    log(f"RESUMED at step {st.step} (epoch {saved['epoch']}{' complete' if saved['complete'] else ''}) from {latest}")

## One micro-step: forward, loss, backward

One case goes through the model, the loss is computed against the judges' distribution, and the backward pass adds its gradients to the LoRA weights' gradients. With accumulation, each micro-step's loss is divided by the number of micro-steps per update, so their gradients add up to an average. A loss that is not a number for 50 micro-steps in a row stops the run.

In [ ]:
def micro_step(model, scaler, items, acc, st):
    """Forward pass on one case, the loss, and the backward pass. The loss is divided by acc so that acc accumulated
    micro-batches add up to one average gradient. A loss that stays not-a-number (an fp16 overflow inside the model)
    for 50 micro-batches in a row on any GPU stops the run."""
    loss = soft_loss(model.scores(items), items)
    scaler.scale(loss / acc).backward()
    st.losses.append(loss.item())
    nonfinite = torch.tensor(float(not math.isfinite(st.losses[-1])), device=DEVICE)
    if WORLD > 1:
        dist.all_reduce(nonfinite)
    st.bad = st.bad + 1 if nonfinite.item() else 0
    if st.bad >= 50:
        raise Collapsed(f"loss not finite for {st.bad} micro-batches in a row at step {st.step}")

## Averaging gradients across GPUs

With two GPUs, each computes gradients on different cases. Before the update, the gradients are summed across the GPUs and divided by two, so both take exactly the same step and their copies of the weights stay identical. That is data parallelism, done by hand for the LoRA weights only.

In [ ]:
def average_gradients(params):
    """With several GPUs, each has gradients from its own cases: they are summed across GPUs and divided by WORLD, so
    every GPU takes the same step and the copies of the model stay identical."""
    if WORLD == 1:
        return
    grads = [p.grad if p.grad is not None else torch.zeros_like(p) for p in params]
    flat = torch._utils._flatten_dense_tensors(grads)
    dist.all_reduce(flat)
    flat /= WORLD
    for p, g in zip(params, torch._utils._unflatten_dense_tensors(flat, grads)):
        p.grad = g

## The optimizer step

The update itself: average the gradients, undo the loss scaling, clip the gradient norm to 1.0 (so one odd case cannot throw the weights far), let AdamW update the weights, and move the learning-rate schedule one step on.

In [ ]:
def optimizer_step(params, opt, sched, scaler, st):
    """One update: average the gradients, undo the loss scaling, clip the gradient norm at 1.0 (the norm before clipping
    is logged), step the optimizer (skipped by the scaler if a gradient overflowed), move the schedule on one step."""
    average_gradients(params)
    scaler.unscale_(opt)
    st.norms.append(float(torch.nn.utils.clip_grad_norm_(params, 1.0)))
    scaler.step(opt); scaler.update(); sched.step()
    opt.zero_grad(set_to_none=True)
    st.step += 1

## Watching a run

Every 100 steps, the mean loss and gradient norm and the time left. At step 400 (or halfway through a short epoch), a collapse check scores part of the dev set: a dev KL still close to a uniform guess means the run has failed, and it is stopped.

In [ ]:
def report_progress(st, epoch, total, acc, t0, start_step):
    """Every 100 steps: mean loss and gradient norm over those steps, the time left, and GPU memory."""
    if st.step % 100 and st.step != total:
        return
    finite = [x for x in st.norms[-100:] if math.isfinite(x)] or [float("nan")]
    minutes_left = (total - st.step) / max(1e-9, (st.step - start_step) / (time.perf_counter() - t0)) / 60
    log(f"  epoch {epoch} step {st.step}/{total} | loss {statistics.mean(st.losses[-100 * acc:]):.4f} | "
        f"grad norm mean {statistics.mean(finite):.2f} max {max(finite):.2f} | ~{minutes_left:.1f} min left | {gpu()}")


def collapse_check(model, step):
    """Early in training: if dev KL is still close to that of a uniform guess, the run has collapsed and is stopped."""
    check = dev_d[:300]
    kl, uniform = metrics(predictions(model, check, MODEL["eval_batch"]))["all"]["kl"], uniform_kl(check)
    log(f"  collapse check at step {step}: dev KL {kl:.4f} vs uniform {uniform:.4f}")
    if kl > uniform - COLLAPSE_MARGIN:
        raise Collapsed(f"dev KL {kl:.4f} at step {step}")
    model.train(True)

## The end of an epoch

The whole dev set is scored. The weights are kept if this epoch's dev KL is within `EPOCH_KL_TOL` (0.005) of the best so far: a later epoch wins unless clearly worse, because the end of the schedule (learning rate near zero) usually gives the better-settled model. Each epoch's adapter is saved too.

In [ ]:
def end_epoch(model, st, epoch, ckdir):
    """After each epoch: score the whole dev set, keep these weights if this epoch's dev KL is within EPOCH_KL_TOL of the
    best so far (a later epoch wins unless clearly worse), and save this epoch's adapter."""
    dev = metrics(predictions(model, dev_d, MODEL["eval_batch"]))["all"]
    st.history.append({"epoch": epoch, **dev})
    if epoch == 1 and dev["kl"] > uniform_kl(dev_d) - COLLAPSE_MARGIN:
        raise Collapsed(f"dev KL {dev['kl']:.4f} after epoch 1")
    st.min_kl = min(st.min_kl, dev["kl"])
    if dev["kl"] <= st.min_kl + EPOCH_KL_TOL:
        st.best = (dev["kl"], {n: p.detach().cpu().clone() for n, p in model.m.named_parameters() if p.requires_grad}, epoch)
    log(f"epoch {epoch} done | dev {dev} | keeping epoch {st.best[2]}")
    if RANK == 0:
        model.m.save_pretrained(ckdir / f"epoch{epoch}_adapter")

## Training one seed

The loop that uses all the pieces: build the optimizer, restore a resume point if there is one, then for every step of every epoch run the micro-steps and the update, report progress, check for collapse, and save resume points. At the end, the kept epoch's weights are put back.

In [ ]:
def train_model(model, seed, ckdir):
    """One seed: the pieces above in a loop over epochs and steps; resumes from ckdir if a resume point is there."""
    acc = max(1, CASES_PER_STEP // WORLD)                # micro-batches per GPU and step: the same cases on 2 GPUs or 1
    per_epoch = len(training_order(seed, 1, acc)) // (WORLD * acc)
    total = EPOCHS * per_epoch
    params = model.lora_params()
    if WORLD > 1:                                        # the same starting weights on every GPU
        for p in params:
            dist.broadcast(p.data, 0)
    torch.manual_seed(seed * 100 + RANK)                 # dropout differs between GPUs
    opt, sched, scaler = make_optimizer(params, model.cfg["lr"], total)
    st = TrainState()
    load_checkpoint(ckdir, model, opt, sched, scaler, st)
    collapse_at = min(COLLAPSE_STEP, max(50, per_epoch // 2))
    log(f"training seed {seed} on {WORLD} GPU(s): {per_epoch} optimizer steps per epoch ({acc} x {WORLD} cases per step), "
        f"{total} steps")
    t0, start_step = time.perf_counter(), st.step
    for epoch in range(st.start_epoch, EPOCHS + 1):
        model.train(True)
        opt.zero_grad(set_to_none=True)
        order = training_order(seed, epoch, acc)
        for s in range(st.step - (epoch - 1) * per_epoch, per_epoch):   # the steps of this epoch not done yet
            for a in range(acc):
                micro_step(model, scaler, prepare(order[(s * acc + a) * WORLD + RANK], seed, epoch), acc, st)
            optimizer_step(params, opt, sched, scaler, st)
            report_progress(st, epoch, total, acc, t0, start_step)
            if st.step == collapse_at:
                collapse_check(model, st.step)
            if st.step % CKPT_EVERY == 0 and st.step % per_epoch:
                save_checkpoint(ckdir, model, opt, sched, scaler, st, epoch, False)
        end_epoch(model, st, epoch, ckdir)
        save_checkpoint(ckdir, model, opt, sched, scaler, st, epoch, True)
        sync_checkpoints(f"seed {seed} epoch {epoch}")
    by_name = dict(model.m.named_parameters())
    for n, v in st.best[1].items():                      # back to the kept epoch's weights
        by_name[n].data.copy_(v.to(by_name[n].device))
    minutes = (time.perf_counter() - t0) / 60
    log(f"training done in {minutes:.1f} min | epoch {st.best[2]} kept (dev KL {st.best[0]:.4f})")
    return {"train_minutes": minutes, "best_epoch": st.best[2], "history": st.history, "gpus": WORLD}

## Run folders

Each seed has its own folder under `checkpoints/`, named after the recipe and the seed. Each epoch's adapter is saved in `attempt-seed<n>/epoch<k>_adapter/` and the kept one in `adapter/`; `02_evaluate.ipynb` reads them from there.


In [ ]:
def run_dir(seed):
    """Where a seed's run lives."""
    short = MODEL["name"].split("/")[-1]
    return CKPT / f"signpost-d-{short}" / f"lr{MODEL['lr']:g}-r{MODEL['r']}-ep{EPOCHS}-seed{seed}"


## Scoring and saving a run

After training, the model is scored on dev as vLLM serves it (no temperature) and, for reference only, with one temperature per question type fitted there; vLLM applies none, so the untempered numbers are the ones the benchmark sees. The results, the dev predictions and the adapter are saved, and pushed with `CKPT_REPO`. `02_evaluate.ipynb` scores the benchmark through vLLM.


In [ ]:
def score_run(model, tag):
    """Our dev, untempered (as vLLM serves it) and with temperatures per type fitted there."""
    batch = MODEL["eval_batch"]
    dev_preds = predictions(model, dev_d, batch)
    temps = fit_temperatures(dev_preds)
    res = {"temperature": temps, "dev": metrics(dev_preds), "dev_tempered": metrics(dev_preds, temps)["all"],
           "dev_by_kind": by_group(dev_preds, None, lambda it: KIND.get((it["id"], it["qid"]), "?"))}
    fmt = lambda d: " | ".join(f"{k} {v['accuracy']}" for k, v in d.items() if k != "all")
    log(f"{tag} dev (as served, no temperature): {res['dev']['all']} | by type: {fmt(res['dev'])} | "
        f"by kind: {fmt(res['dev_by_kind'])}")
    log(f"{tag} dev with temperatures {temps}: {res['dev_tempered']}")
    return res, dev_preds


def finish(model, rdir, tag, res, dev_preds, adapter):
    if RANK == 0:
        rdir.mkdir(parents=True, exist_ok=True)
        json.dump(res, open(rdir / "results.json", "w"), indent=1)
        dump = [{"id": it["id"], "qid": it["qid"], "p": [round(p[n], 4) for n in it["options"]]} for it, p in dev_preds]
        json.dump(dump, open(rdir / "dev_preds.json", "w"))
        if adapter:
            model.m.save_pretrained(rdir / "adapter")
            for a in rdir.glob("attempt-seed*/latest.pt"):
                a.unlink()
        json.dump({k: res[k]["all"] for k in ("dev",)}, open(rdir / "done.json", "w"),
                  indent=1)
        sync_checkpoints(f"{tag} finished")
    barrier()

## Every seed

Finished seeds are read back. A seed that collapses twice is skipped.


In [ ]:
def run_all():
    for seed in SEEDS:
        rdir, tag = run_dir(seed), f"seed {seed}"
        if (rdir / "done.json").exists():
            log(f"===== {tag}: finished earlier, read back from {rdir}: {json.load(open(rdir / 'done.json'))['dev']}")
            continue
        log(f"===== {tag} ===== | {WORLD} GPU(s)")
        for attempt in range(ATTEMPTS):
            s = seed + 1000 * attempt
            random.seed(s); torch.manual_seed(s)               # the same LoRA start on every GPU
            model = DecoderModel(MODEL, train=True)
            try:
                res = {"seed": seed, "attempt_seed": s, "trained": train_model(model, s, rdir / f"attempt-seed{s}")}
                break
            except Collapsed as e:
                log(f"attempt {attempt + 1} COLLAPSED: {e}")
                model.free()
        else:
            log(f"{tag}: collapsed in every attempt; skipped")
            continue
        scores, dev_preds = score_run(model, tag)
        finish(model, rdir, tag, {**res, **scores}, dev_preds, adapter=True)
        model.free()


def worker():
    """One process: its GPU from the launcher's environment, then the whole loop."""
    global RANK, WORLD, DEVICE
    WORLD, RANK = int(os.environ.get("WORLD_SIZE", "1")), int(os.environ.get("RANK", "0"))
    if WORLD > 1:
        DEVICE = torch.device(f"cuda:{int(os.environ['LOCAL_RANK'])}")
        torch.cuda.set_device(DEVICE)
        dist.init_process_group("nccl", rank=RANK, world_size=WORLD, timeout=datetime.timedelta(hours=1), device_id=DEVICE)
    try:
        run_all()
    finally:
        if WORLD > 1:
            dist.destroy_process_group()

## Run

With two GPUs the work starts in two processes through `accelerate`'s `notebook_launcher` (no cell above may use CUDA); with one GPU it runs here.

In [ ]:
N_GPU = min(GPUS, torch.cuda.device_count()) if AMP else 1
log(f"starting on {N_GPU} GPU(s)")
if N_GPU > 1:
    assert not torch.cuda.is_initialized(), "CUDA was started in this kernel before the launch: restart the kernel and run all"
    from accelerate import notebook_launcher
    notebook_launcher(worker, num_processes=N_GPU)
else:
    worker()

## Result

`02_evaluate.ipynb` merges the three seeds' epoch-2 adapters into the final model: the mean of their weight changes, added to the base at 0.75. On Typed Decisions (test split, 2,000 decisions, general mode, scored on HF Jobs with the benchmark's own scorer) it scores **accuracy 0.712, KL 0.1615, Brier 0.0853, ECE 0.0914**. The published model is [skamalj/signpost-d-4b](https://huggingface.co/skamalj/signpost-d-4b).
